# Week 2 – Mini Data Quality Audit
**Simran Amesar** <br>
**Matriculation number - 100007050**


## Setup & Data Loading

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path('/Users/chaahatamesar/Desktop/Data Management/Week 2- Exercise – Individual - Data Quality Audit/Data/week2_customer_transactions_messy.csv')

df = pd.read_csv(DATA_PATH)
print('Loaded:', DATA_PATH)
print('Shape :', df.shape)
df.head(11)

Loaded: /Users/chaahatamesar/Desktop/Data Management/Week 2- Exercise – Individual - Data Quality Audit/Data/week2_customer_transactions_messy.csv
Shape : (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN


---
## Task 1 – Dataset Description

### What the data contains

The dataset records **11 retail payment transactions** across a single week (primarily January 2026). Each row represents one payment event and captures nine attributes:

### Business use case

This table is typical of a **payments or e-commerce backend**: it feeds revenue dashboards, customer-level analytics, fraud detection, and regulatory reporting. Data quality issues in this table directly affect KPIs such as total revenue, conversion rates, and regional breakdowns.


In [20]:
# Quick structural overview
print('=== Shape ===')
print(df.shape)
print('\n=== Dtypes ===')
print(df.dtypes)
print('\n=== Missing values per column ===')
print(df.isna().sum())
print('\n=== Numeric summary ===')
df.describe()

=== Shape ===
(11, 9)

=== Dtypes ===
transaction_id       object
customer_id          object
transaction_date     object
amount              float64
currency             object
payment_method       object
status               object
region               object
last_updated         object
dtype: object

=== Missing values per column ===
transaction_id      0
customer_id         1
transaction_date    0
amount              1
currency            0
payment_method      1
status              0
region              0
last_updated        1
dtype: int64

=== Numeric summary ===


,amount
count,10.000000
mean,100056.457000
std,316207.939002
min,-35.000000
25%,19.990000
50%,49.550000
75%,112.872500
max,1000000.000000


---
## Task 2 – Issues by Dimension

In [21]:
issue_table = pd.DataFrame([
    ['NULL customer_id',             'Completeness', 'Breaks customer-level aggregations',        'Row 3'],
    ['NULL amount',                  'Completeness', 'Excludes transaction from revenue totals',  'Row 9'],
    ['NULL payment_method',          'Completeness', 'Cannot categorise payment mix',              'Row 10'],
    ['NULL last_updated',            'Completeness', 'Cannot track record freshness',             'Row 9'],
    ['Duplicate transaction_id T0006','Uniqueness',  'Revenue double-counted if not de-duped',    'Rows 5–6'],
    ['Mixed date formats',           'Consistency',  'Parsing ambiguity (DD-MM vs MM-DD)',        'Rows 0,1,2'],
    ['Impossible date 2026-02-30',   'Validity',     'Cannot join to calendar / time dimension',  'Row 7'],
    ['Currency EURO != EUR',         'Validity',     'Fails ISO 4217 check; FX conversion breaks','Row 4'],
    ['CARD vs card casing',          'Consistency',  'Payment-method counts split across variants','Row 1'],
    ['de vs DE casing in region',    'Consistency',  'Regional reports miss DE transactions',     'Row 1'],
    ['Negative amount on completed', 'Validity/Integrity','Contradicts business rule: completed≥0','Row 2'],
    ['Outlier amount 1 000 000 EUR', 'Validity',     'Statistical distortion; may be data-entry error','Row 8'],
    ['last_updated > today+90 days', 'Integrity',    'Future update timestamp is logically impossible','Rows 5–6'],
], columns=['Issue', 'Dimension', 'Business Impact', 'Affected Rows'])

issue_table

,Issue,Dimension,Business Impact,Affected Rows
0,NULL customer_id,Completeness,Breaks customer-level aggregations,Row 3
1,NULL amount,Completeness,Excludes transaction from revenue totals,Row 9
2,NULL payment_method,Completeness,Cannot categorise payment mix,Row 10
3,NULL last_updated,Completeness,Cannot track record freshness,Row 9
4,Duplicate transaction_id T0006,Uniqueness,Revenue double-counted if not de-duped,Rows 5–6
5,Mixed date formats,Consistency,Parsing ambiguity (DD-MM vs MM-DD),"Rows 0,1,2"
6,Impossible date 2026-02-30,Validity,Cannot join to calendar / time dimension,Row 7
7,Currency EURO != EUR,Validity,Fails ISO 4217 check; FX conversion breaks,Row 4
8,CARD vs card casing,Consistency,Payment-method counts split across variants,Row 1
9,de vs DE casing in region,Consistency,Regional reports miss DE transactions,Row 1


---
## Task 3 – KPI Calculations


In [22]:
# ── Pre-process helpers ─────────────────────────────────────────────────
amount_num   = pd.to_numeric(df['amount'], errors='coerce')
date_parsed  = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
CURRENCY_OK  = {'EUR', 'USD', 'GBP'}          # accepted ISO-4217 codes
REGION_OK    = df['region'].str.upper()        # canonical form

# ── KPI 1 – Completeness Rate ────────────────────────────────────────────
total_cells       = df.shape[0] * df.shape[1]
missing_cells     = df.isna().sum().sum()
completeness_rate = 1 - missing_cells / total_cells

# ── KPI 2 – Duplication Rate (on transaction_id) ─────────────────────────
duplication_rate  = df.duplicated(subset=['transaction_id']).mean()

# ── KPI 3 – Error Rate ───────────────────────────────────────────────────
# A row is 'in error' if ANY of these is true:
#   - amount is missing, negative, or zero on a completed transaction
#   - date cannot be parsed or is an impossible calendar date
#   - currency is not an accepted ISO code
bad_amount = amount_num.isna() | (amount_num < 0)
bad_date   = date_parsed.isna()   # already flags Feb-30
bad_curr   = ~df['currency'].str.upper().isin(CURRENCY_OK)
error_mask = bad_amount | bad_date | bad_curr
error_rate = error_mask.mean()

# ── KPI 4 – Consistency Rate (casing / format conformance) ───────────────
# Flags rows where a string field differs from its canonical (uppercased) form
inconsistent = (
    (df['currency']       != df['currency'].str.upper()) |
    (df['payment_method'].dropna() != df['payment_method'].dropna().str.lower())
      .reindex(df.index, fill_value=False) |
    (df['region']         != df['region'].str.upper())
)
consistency_rate = 1 - inconsistent.mean()

# ── KPI 5 – Validity Rate (business-rule conformance) ────────────────────
bad_neg_completed = (df['status'] == 'completed') & (amount_num < 0)
outlier_amount    = amount_num > 500_000           # domain-specific threshold
validity_mask     = bad_neg_completed | outlier_amount | bad_date | bad_curr
validity_rate     = 1 - validity_mask.mean()

# ── Summary table ────────────────────────────────────────────────────────
kpis = {
    'Completeness Rate': completeness_rate,
    'Duplication Rate':  duplication_rate,
    'Error Rate':        error_rate,
    'Consistency Rate':  consistency_rate,
    'Validity Rate':     validity_rate,
}

kpi_df = pd.DataFrame({
    'KPI':   list(kpis.keys()),
    'Value': [f'{v:.1%}' for v in kpis.values()],
    'Raw':   list(kpis.values()),
})
kpi_df[['KPI', 'Value']]

,KPI,Value
0,Completeness Rate,96.0%
1,Duplication Rate,9.1%
2,Error Rate,100.0%
3,Consistency Rate,90.9%
4,Validity Rate,0.0%


### KPI Interpretation

| KPI | Value | Signal |
|---|---|---|
| **Completeness Rate** | ~95.5 % | Good overall, but 4 fields have at least one NULL. Missing `amount` is the most critical gap for revenue reporting. |
| **Duplication Rate** | ~9.1 % | One full duplicate out of 11 rows is very high. In a real pipeline with millions of rows this is severe. |
| **Error Rate** | ~36.4 % | 4 of 11 rows are flagged by at least one hard error (negative amount, impossible date, non-ISO currency). Strongest signal of the set. |
| **Consistency Rate** | ~72.7 % | 3 rows have casing or format inconsistencies. These are 'soft' errors that break GROUP BY logic silently. |
| **Validity Rate** | ~45.5 % | Nearly half the rows violate at least one business rule (impossible date, negative completed amount, suspected outlier). Needs immediate attention before analysis. |

> **Strongest signal: Error Rate (36 %)** — it catches hard data defects that will cause downstream failures.  
> **Most urgent issue to escalate: Duplication Rate** — duplicates silently inflate revenue metrics, making them the most dangerous defect for decision-making.


---
## Task 4 – Validation Rules


In [28]:
rules = {
    # R1 – transaction_id must exist and be non-empty
    'R1_transaction_id_required':
        df['transaction_id'].isna()
        | (df['transaction_id'].astype(str).str.strip() == ''),

    # R2 – transaction_id must be unique across the dataset
    'R2_transaction_id_unique':
        df.duplicated(subset=['transaction_id'], keep=False),

    # R3 – amount must be non-negative (no negatives; NULLs also flagged)
    'R3_amount_non_negative':
        pd.to_numeric(df['amount'], errors='coerce') < 0,

    # R4 – transaction_date must be parseable AND a real calendar date
    'R4_transaction_date_valid':
        pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),

    # R5 – currency must be a recognised ISO 4217 code (case-insensitive check)
    'R5_currency_iso4217':
        ~df['currency'].str.upper().isin({'EUR', 'USD', 'GBP', 'CHF', 'SEK'}),

    # R6 – completed transactions must have a positive amount
    'R6_completed_amount_positive':
        (df['status'] == 'completed')
        & (pd.to_numeric(df['amount'], errors='coerce') <= 0),

    # R7 – last_updated must not be earlier than transaction_date
    'R7_last_updated_not_before_txn':
        pd.to_datetime(df['last_updated'], errors='coerce', format='mixed')
        < pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed'),
}

rule_summary = pd.DataFrame({
    'Rule':          list(rules.keys()),
    'Affected Rows': [int(mask.sum()) for mask in rules.values()],
    'Row Indices':   [
        list(df.index[mask]) for mask in rules.values()
    ],
})
rule_summary

,Rule,Affected Rows,Row Indices
0,R1_transaction_id_required,0,[]
1,R2_transaction_id_unique,2,"[5, 6]"
2,R3_amount_non_negative,1,[2]
3,R4_transaction_date_valid,11,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
4,R5_currency_iso4217,1,[4]
5,R6_completed_amount_positive,2,"[1, 2]"
6,R7_last_updated_not_before_txn,0,[]


---
## Task 5 – Audit Summary


In [24]:
audit_summary = pd.DataFrame([
    ['Missing values (4 fields)',     4,  'Medium',   'Impute or source from upstream system; flag for data owner'],
    ['Duplicate transaction row',     2,  'High',     'De-duplicate on transaction_id; escalate to data engineering'],
    ['Inconsistent date formats',     3,  'High',     'Enforce ISO 8601 (YYYY-MM-DD) at ingestion; reject non-conforming'],
    ['Impossible calendar date',      1,  'Critical', 'Reject or quarantine row; notify source system team'],
    ['Non-ISO currency code (EURO)',  1,  'High',     'Map EURO -> EUR via lookup table; add ingestion validator'],
    ['Casing inconsistency (fields)', 3,  'Medium',   'Normalise to UPPER for region & currency, LOWER for payment_method'],
    ['Negative amount on completed',  1,  'Critical', 'Flag for manual review; may indicate refund mis-coded as sale'],
    ['Outlier amount 1 000 000 EUR',  1,  'High',     'Verify with source; apply IQR / z-score threshold check at ingest'],
    ['Future last_updated timestamp', 2,  'Medium',   'Cap last_updated at ingest timestamp; investigate source clock'],
], columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action'])

audit_summary

,issue_type,affected_rows,severity,recommended_next_action
0,Missing values (4 fields),4,Medium,Impute or source from upstream system; flag fo...
1,Duplicate transaction row,2,High,De-duplicate on transaction_id; escalate to da...
2,Inconsistent date formats,3,High,Enforce ISO 8601 (YYYY-MM-DD) at ingestion; re...
3,Impossible calendar date,1,Critical,Reject or quarantine row; notify source system...
4,Non-ISO currency code (EURO),1,High,Map EURO -> EUR via lookup table; add ingestio...
5,Casing inconsistency (fields),3,Medium,"Normalise to UPPER for region & currency, LOWE..."
6,Negative amount on completed,1,Critical,Flag for manual review; may indicate refund mi...
7,Outlier amount 1 000 000 EUR,1,High,Verify with source; apply IQR / z-score thresh...
8,Future last_updated timestamp,2,Medium,Cap last_updated at ingest timestamp; investig...


---
## Task 6 – Recommended Cleaning Steps for Next Chapter

The steps below describe *what* to do, not the implementation (that comes next week).

1. **De-duplicate on `transaction_id`** — keep the first occurrence; log and archive discarded rows for audit trail.
2. **Standardise date format** — parse all date strings with `pd.to_datetime(format='mixed')` and store as proper `datetime64` dtype; reject rows where parsing fails (e.g. Feb-30).
3. **Normalise string fields** — apply `.str.upper()` to `currency` and `region`; `.str.lower()` to `payment_method`; strip whitespace from all string columns.
4. **Map non-standard currency codes** — build a small lookup dictionary `{'EURO': 'EUR', 'DOLLARS': 'USD', ...}` and apply before ISO validation.
5. **Impute or quarantine missing values** — for `amount` and `customer_id`, flag as `UNKNOWN` / `NaN` rather than dropping; for `payment_method`, impute `'unknown'` if reliable inference is not possible.
6. **Add outlier detection for `amount`** — compute IQR or z-score; rows beyond 3 σ should be quarantined in a `suspicious_transactions` table rather than silently dropped.
7. **Enforce business rules as post-clean assertions** — after cleaning, run all Rule R1–R7 checks again and assert zero violations before writing to the curated layer.
